In [8]:
# -*- coding: utf8 -*-
import os
from os.path import expanduser
import itertools
import yaml,YamlDuplicates
from yaml.constructor import ConstructorError
import warnings
import ParFuMor as PFM
from ParFuMor import *
import pickle
from IPython.display import HTML, display
#import cellbell 

In [9]:
%store -r numerosKalaba typeKalaba 
%store -r anneeKalaba

# anneeKalaba=25
numerosKalaba=[1,2,3,4,5]
numerosKalaba=[6]
print numerosKalaba
typeKalaba="Kalaba"
%store numerosKalaba 
%store anneeKalaba
%store typeKalaba

[6]
Stored 'numerosKalaba' (list)
Stored 'anneeKalaba' (int)
Stored 'typeKalaba' (str)


In [10]:
home = expanduser("~")
repertoire=home+"/sDrive/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/"
annee=anneeKalaba
if typeKalaba!="Kanonik":
    serie=repertoire+"%d-"%annee
    nomsKalabas=[serie+"K%d/"%num for num in numerosKalaba]
else:
    serie=repertoire+"%d-Kanoniks/"%annee
    nomsKalabas=[serie+"Kanonik-%02d/"%num for num in numerosKalaba]

In [11]:
class HierarchieCF:
    '''
    hiérarchie des classes flexionnelles
    '''
    def __init__(self):
        self.classes={}
        self.superieur={}
        self.categorie={}
        self.trait={}
        self.sets={}
        self.inherents={}

    def addCategory(self,superclasse,classe):
        if not superclasse in self.classes:
            self.classes[superclasse]=[]
        self.classes[superclasse].append(classe)
        self.superieur[classe]=superclasse
        if superclasse in gloses:               #si superclasse est une catégorie
            self.categorie[classe]=superclasse
            if not superclasse in self.inherents:
                self.inherents[superclasse]=[]
            if not classe in self.inherents[superclasse]:
                self.inherents[superclasse].append(classe)
            category=superclasse
        else:                                   #si superclasse est une classe flexionnelle
            self.categorie[classe]=self.categorie[superclasse]
            if not classe in self.inherents[self.categorie[classe]]:
                self.inherents[self.categorie[classe]].append(classe)
            category=self.categorie[superclasse]
        noFeature=True
        for element in self.sets[category]:
            for featureSet in element:
                for valeur in featureSet.split(","):
                    if classe == valeur:
                        hierarchieCF.addFeature(category,classe,element[featureSet])
                        noFeature=False
        if noFeature:
            hierarchieCF.addFeature(category,classe,"CF")

    def addFeatureSet(self,category,attribute,values):
        if not category in self.sets:
            self.sets[category]=[]
        self.sets[category].append({values:attribute})

    def addFeature(self,category,classe,feature):
        if not category in self.trait:
            self.trait[category]=[]
        if not {classe:feature} in self.trait[category]:
            self.trait[category].append({classe:feature})

    def getFeature(self,category,classe):
        if category in self.trait:
            return [x for x in self.trait[category] if classe in x.keys()][0][classe]
        else:
            return "ClassFLex"

    def categoryLookup(self,categorie):
        if categorie in gloses:
            return categorie
        else:
            return self.categoryLookup(self.categorie[categorie])

    def getCategory(self,classe):
        '''
        donne la catégorie correspondant à une classe ou à une catégorie
        '''
        if classe in self.categorie:
            return self.categorie[classe]
        elif classe in self.classes:
            return classe
        else:
            return classe

In [13]:
with open(serie+"K6/"+"Stems.yaml", 'r') as stream:
    try:
        stems=yaml.safe_load(stream)
        PFM.stems=stems
    except ConstructorError,msg:
        print msg
